# Chapter 1

### Encoder Decoder architecture in keras

```
### GRU Layers
# return_sequences=True will return all 10 unit states in a GRU layer while return_state=True will only return the final state
gru_layer_simple = keras.layers.GRU(10)
gru_tensors = gru_layer_simple(input_tensors) 
model = keras.models.Model(inputs=input_tensors, outputs=gru_tensors)

gru_layer_return_state = keras.layers.GRU(10, return_state=True)
gru_tensors, gru_state = gru_layer_return_state(input_tensors )

gru_layer_return_sequences = keras.layers.GRU(10, return_sequences=True)
gru_tensors, gru_sequence = gru_layer_simple(input_tensors, return_sequences=True) 

### RepeatVector Layers
from tensorflow.keras.layers import RepeatVector
input_tensors = Input(shape=(3,)) # Input of unknown records (rows) and 3 dimensional onehot vector
repeat_layer = RepeatVector(5) # How many words (columns) are we thinking of generating 
repeat_tensors = repeat_layer(input_tensors) # Will be of shape (None, 5, 3)
repeat_model = Model(inputs=input_tensors, outputs=repeat_tensors)

################## Encoder Decoder architecture ###############
# Encoder example : from words to categorical encoding (Encoder GRU consumes words and outputs context vector)
# Decoder example : from categorical encoding to words (Decoder GRU consumes context vector and outputs sequence of outputs )
# Encoder-Decoder : output state of encoder is connected with the initial state of a decoder (Decoder prediction layer sits on top of decoder, consumes decoder GRU outputs and produces prediction probabilities of sequence with Timedistributed Dense layer )

from keras.layers import Dense, TimeDistributed, Input, GRU, RepeatVector
# Designing encoder model : takes in a sentence of n length and consumes the vocabulary of m tokens
encoder_inputs = Input(shape=(sentence_len, unique_words)) # sentence_len is length of words in sentence and unique_words is vocabulary
encoder_gru_layer = GRU(units, return_state=True) # this layer will take inputs and create a single vector
encoder_output_tensors, encoder_state = encoder_gru_layer(encoder_inputs)
encoder_model = Model(inputs=encoder_inputs, outputs=encoder_state) 

# Designing decoder model : 
repeat_layer = RepeatVector(n) # This layer will generate n number of words and create column dimension (n is same as sentence_len)
decoder_inputs = repeat_layer(encoder_state) # encoder_state contains 1 value for 1 word, with repeat_layer we repeat that for n words
decoder_gru_layer = GRU(units, return_sequences=True) # this layer will take a series of vectors and generate a new series of vectors
decoder_output_tensors = decoder_gru_layer(decoder_inputs, initial_state=encoder_state) # Connecting final state of encoder with first gru layer of decoder (repeat_layer only repeats the process, the original sequence calculation is done in GRU layer)
decoder_model = Model(inputs=encoder_inputs, outputs=decoder_output_tensors)

# Putting a time-distributed dense layer on top of decoder to get predictions based on time-sequence
from keras.initializers import RandomNormal
init = RandomNormal()
dense_layer = Dense(unique_words, activation='softmax', kernel_initializer=init, bias_initializer=init) # A dense layer
timed_dense_layer = TimeDistributed(dense_layer) # wrapping the dense layer with a time-sequence layer
final_tensors = timed_dense_layer(decoder_output_tensors)
encoder_decoder_model = keras.models.Model(inputs=encoder_inputs, outputs=final_tensors)
encoder_decoder_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['acc'])
print(model.summary()) # See summary

y = model.predict(x) # y is a tensor with multiple probability value on the last dimension
classes = np.argmax(y, axis=-1) # Take the index where the max value resides to filter the most probable class

# NOTE : Also see TEACHER FORCING IN KERAS
```

### Text processing in keras

```
### Text preprocessing
# Tokenization : breaking a sentence into individual words
from tensorflow.keras.preprocessing.text import Tokenizer
en_tok = Tokenizer(num_words=50, oov_token='UNK')  # limit impact of rare words, replace with oov token for unseen token with 'UNK'
en_tok.fit_on_texts(en_text) 
id = en_tok.word_index["january"] # returns index of the word in the tokenized list (eg : 51)
w = en_tok.index_word[51] # returns word of specified index in the tokenized list (eg: 'january')
sentences = ["I love to play football", "He loves to play cricket"]
seqs = en_tok.texts_to_sequences(sentences) # Transforming word sequence into  tokens sequence (eg: [[26, 70, 27, 73, 7], ...])

# Padding: Add pads at the beginning or end of sequence or truncate words from the beginning or the end of sequence
from tensorflow.keras.preprocessing.sequence import pad_sequences
preproc_text = pad_sequences(seqs, padding='post', truncating='post', maxlen=9) # [18, 20, 2, 10] => [18 20 2 10 0 0 0 0 0]
pad_seq = pad_seq[:,::-1] # Reverse the sequence for stronger connection between encoder and decoder

shuffled_df = df.sample(frac=1, random_state=42).reset_index(drop=True) # Shuffle dataframe
# Make sure to add "sos" at the beginning of each sentence and "eos" at the end of each sentence (use for loop and append if required)
# Splitting data into 
train_size, valid_size = 800, 200
train_data = df.values[:train_size]     # Do this on both en_text and fr_text
valid_data = df.values[train_size:train_size+valid_size]     # Do this on both en_text and fr_text

from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

en_len = 15 # number of words in a sentence
en_vocab = 150 # Total unique words found the all the text

## Helper function that takes in text and outputs into padded, embedded, reversed, one-hot vectors 
def sents2seqs(input_type, sentences, onehot=False, pad_type='post', reverse=False):     
    encoded_text = en_tok.texts_to_sequences(sentences) # Convert sentences to embedded sequences   
    preproc_text = pad_sequences(encoded_text, padding=pad_type, truncating='post', maxlen=en_len) # Padding for consistent length 
    if reverse: # Reverse the text using numpy axis reversing
      preproc_text = preproc_text[:, ::-1]
    if onehot: # Convert the word IDs to onehot vectors
        preproc_text = to_categorical(preproc_text, num_classes=en_vocab)
    return preproc_text
# Call sents2seqs to get the padded and reversed sequence of IDs
sentences = ["It is never rainy during july ."]
pad_seq = sents2seqs('source', sentences, reverse= True)
rev_sent = [en_tok.index_word[wid] for wid in pad_seq[0][-6:]] 
print('Reversed: ',' '.join(rev_sent)) # Get full sentence by joining with ' '

# Build  an encoder decoder model (NOTE : See ENCODER DECODER IN KERAS)

from keras.layers import Input, GRU, RepeatVector, TimeDistributed, Dense
from keras.models import Model

input_shape = (en_len, en_vocab)  
encoder_input = Input(shape=input_shape)
encoder_gru = GRU(48, name='gru', return_state=True)(encoder_input)
encoder_output, encoder_state = encoder_gru

decoder_input = RepeatVector(15)(encoder_state)
decoder_gru = GRU(48, return_sequences=True)(decoder_input, initial_state=encoder_state)
decoder_output = TimeDistributed(Dense(en_vocab, activation='softmax'))(decoder_gru)
model = Model(inputs=encoder_input, outputs=decoder_output)

# Train in batches
n_epochs, b_size = 5, 250 # Epoch and batch size
for ei in range(n_epochs):
    for i in range(0,data_size,b_size): # NOTE: all these _df are actually dataframe converted into list
        en_x = sents2seqs('source', train_df_en[i:i+b_size], onehot=True, pad_type='pre') # Source language
        de_y = sents2seqs('target', train_df_fr[i:i+b_size], onehot=True) # Translated language
        model.train_on_batch(en_x, de_y)
    v_en_x = sents2seqs('source', validation_df_en, onehot=True, pad_type='pre')
    v_de_y = sents2seqs('target', validation_df_fr, onehot=True)
    res = model.evaluate(v_en_x, v_de_y, batch_size=valid_size, verbose=0)
    res[0] # Loss
    res[1]*100.0 # Accuracy

# Sanity checks for debugging
en_st = ['it is sometimes chilly during decemberand freezing in june .']
en_seq = sents2seqs('source', en_st, onehot=True, reverse=True) # Transform the encoder sentence
np.argmax(en_seq, axis=-1)
fr_pred = model.predict(en_seq)
fr_pred.shape # [num sentences, sequence len, vocab size]
fr_seq = np.argmax(fr_pred, axis=-1)[0] # [[ 3 7 35 34 2 ... 5 4 4 0 0]] # take the max probability value from vocabulary dimension 
fr_seq.shape  # [num sentences, sequence len]
word_list = [fr_tok.index_word[i] for i in fr_seq if i != 0] # Filter out the words with 0 probabilities and create word list
fr_sentence = ' '.join(word_list) # Join words to get complete sentence

# NOTE : Also see TEACHER FORCING IN KERAS
```

### Encoder Decoder Architecture

<center><img src="images/01.01.png"  style="width: 400px, height: 300px;"/></center>
<center><img src="images/01.02.png"  style="width: 400px, height: 300px;"/></center>


### GRU Layer

<center><img src="images/01.03.png"  style="width: 400px, height: 300px;"/></center>
<center><img src="images/01.05.png"  style="width: 400px, height: 300px;"/></center>
<center><img src="images/01.06.png"  style="width: 400px, height: 300px;"/></center>
<center><img src="images/01.04.png"  style="width: 400px, height: 300px;"/></center>


# Chapter 2

### Encoder Decoder with TimeDistributed Dense Layer

<center><img src="images/02.01.png"  style="width: 400px, height: 300px;"/></center>
<center><img src="images/02.02.png"  style="width: 400px, height: 300px;"/></center>
<center><img src="images/02.03.png"  style="width: 400px, height: 300px;"/></center>
<center><img src="images/02.04.png"  style="width: 400px, height: 300px;"/></center>


# Chapter 4

### Teacher Forcing

- An encoder decoder model
- The only difference is that the decoder will take input 1 less word
- The goal is to predict the sequence with missing word 

<center><img src="images/04.01.png"  style="width: 400px, height: 300px;"/></center>
<center><img src="images/04.02.png"  style="width: 400px, height: 300px;"/></center>
<center><img src="images/04.03.png"  style="width: 400px, height: 300px;"/></center>
<center><img src="images/04.04.png"  style="width: 400px, height: 300px;"/></center>
<center><img src="images/04.05.png"  style="width: 400px, height: 300px;"/></center>

```
# An encoder decoder model
# The only difference is that the decoder will take input 1 less word
# The goal is to predict the sequence with missing word 
# Build  an encoder decoder model (NOTE : See ENCODER DECODER IN KERAS)

from keras.layers import Input, GRU, RepeatVector, TimeDistributed, Dense
from keras.models import Model

# Encoder
en_input_tensors = Input(shape=(en_len, en_vocab))
en_gru_layer = GRU(hsize, return_state=True)
en_output_tensors, en_state = en_gru_layer(en_input_tensors)

# Decoder
de_input_tensors = Input(shape=(fr_len-1, fr_vocab)) # takes 1 word less than encoder
de_gru_layer = GRU(hsize, return_sequences=True)
de_output_tensors = de_gru_layer(de_input_tensors, initial_state=en_state)

# Decoder Prediction
de_dense_layer = Dense(fr_vocab, activation= 'softmax' )
de_timed_dense_layer = TimeDistributed( de_dense_layer ) 
de_prediction = de_timed_dense_layer(de_output_tensors)

nmt_tf = Model(inputs=[en_input_tensors, de_input_tensors], outputs=de_prediction)
nmt_tf.compile(optimizer='adam', loss="categorical_crossentropy", metrics=["acc"])


for ei in range(n_epochs):
    for i in range(0,train_size,bsize):
        encoder_x = sents2seqs('source', tr_en[i:i+bsize], onehot=True, reverse=True)
        decoder_xy = sents2seqs('target', tr_fr[i:i+bsize], onehot=True)
        decoder_x = decoder_xy[:,:-1,:] # Inputs - All French words except the last word (onehot encoded)
        decoder_y = decoder_xy[:,1:,:] # Outputs - All French words except the first word as next sequence of input (onehot encoded)
        nmt_tf.train_on_batch([en_x, de_x], de_y)
```

### decoder of the inference model (Recursive Decoder that generates word for single timestep)

- one unit of decoder works recursively
- takes input state and input word (onehot) from previous timestep of decoder
- GRU layer then generates output state and output for next timestep of decoder
- output is passed to Dense layer for predicting the most probable output word
- takes tokens 'sos' and 'eos' to understand the beginning and ending of a sentence

<center><img src="images/04.06.png"  style="width: 400px, height: 300px;"/></center>
<center><img src="images/04.07.png"  style="width: 400px, height: 300px;"/></center>


```
### decoder of the inference model (Recursive decoder generator)

import tensorflow.keras.layers as layers
from tensorflow.keras.models import Model

en_input_tensors = Input(shape=(en_len,en_vocab))
en_gru_layer = GRU(hsize, return_state=True)
en_output_tensors, en_state = en_gru_layer(en_input_tensors)
encoder = Model(inputs=en_input_tensors, outputs=en_state) # Our encoder model will generate  output state from input data 

de_input_tensors = Input(shape=(1, fr_vocab)) # An input layer that accepts a single onehot encoded word
de_state_in = Input(shape=(hsize,)) # Takes en_state as input (comes from en_gru_layer same tensor dimension)
de_gru_layer = GRU(hsize, return_state=True) # GRU layer that produces output and a new state
de_output_tensors, de_state_out = de_gru_layer(de_input_tensors, initial_state=de_state_in) # Recursion: state is another input
de_dense_layer = layers.Dense(fr_vocab, activation='softmax') # prediction of most probable word among unique words in fr_vocab
de_predictions = de_dense_layer(de_output_tensors)
decoder = Model(inputs=[de_input_tensors, de_state_in], outputs=[de_predictions, de_state_out])

### You need to copy the weights from the trained model to the inference model. 
### Always remember to perform this step if you have different training and inference models. 
### If you miss this step, the model will still run but the translations will be incorrect.

en_gru_w = tr_en_gru.get_weights() # Load the weights to the encoder GRU from the trained model
en_gru.set_weights(en_gru_w) # Set the weights of the encoder GRU of the inference model
de_gru.set_weights(tr_de_gru.get_weights()) # Load and set the weights to the decoder GRU
de_dense.set_weights(tr_de_dense.get_weights()) # Load and set the weights to the decoder Dense

en_sent = ['the united states is sometimes chilly during december , but it is sometimes freezing in june .']
en_data = sents2seqs('source', en_st, onehot=True, reverse=True) # preprocess the sentence (tokenized, reversed, one-hot encoded)
de_input_state = encoder.predict(en_data) # use encoder model to convert into decoder input state

de_data = word2onehot(fr_tokens, 'sos', fr_vocab) # Converting "sos" (initial word) to a sequence
fr_sent = ''
for _ in range(fr_len): # Keep doing this until the end of the sentence is reached 
    de_prob, de_input_state = decoder.predict([de_data,de_input_state]) # This is where de_input_state keeps changing for each word
    de_word = probs2word(de_prob, fr_tokens) # get the max probable word
    de_data = word2onehot(fr_tokens, de_w, fr_vocab)
    if (de_word == 'eos'): # Stop generating once end of sentence word "eos" is found
        break 
    else:
        fr_sent += de_word + ' ' # Keep adding words to make a sentence

def word2onehot(tokenizer, word, vocab_size):
    de_seq = tokenizer.texts_to_sequences([[word]])
    de_onehot = to_categorical(de_seq, num_classes=vocab_size)
    de_onehot = np.expand_dims(de_onehot, axis=1)    
    return de_onehot

def probs2word(probs, tok):
    wid = np.argmax(probs[0,:], axis=-1)
    w = tok.index_word[wid]
    return w
```

### Word Embeddings (With teacher forcing)

<center><img src="images/04.08.png"  style="width: 400px, height: 300px;"/></center>
<center><img src="images/04.09.png"  style="width: 400px, height: 300px;"/></center>
<center><img src="images/04.10.png"  style="width: 400px, height: 300px;"/></center>
<center><img src="images/04.11.png"  style="width: 400px, height: 300px;"/></center>
<center><img src="images/04.12.png"  style="width: 400px, height: 300px;"/></center>


- Embedding layer creates matrix tensors
- It has a single vector for each word in the vocabulary (unique word in total text)
    - eg: for 100 word vocabulary, if the provided embedding vector size is 96, then the matrix is of size 100X96
- output of embedding layer is passed to GRU layer


```
# The only difference from encoder decoder model is that the decoder will take input 1 less word for teacher forcing
# The goal is to predict the sequence with missing word  (NOTE : See ENCODER DECODER IN KERAS)
from keras.layers import Input, GRU, RepeatVector, TimeDistributed, Dense
from keras.models import Model
### WITHOUT EMBEDDING LAYER

# Encoder
en_input_tensors = Input(shape=(en_len, en_vocab)) # en_len is the number of words in a sentence, en_vocab is no of unique words
en_gru_layer = GRU(hsize, return_state=True)
en_output_tensors, en_state = en_gru_layer(en_input_tensors)
# Decoder
de_input_tensors = Input(shape=(fr_len-1, fr_vocab)) # takes 1 word less than encoder (for different language sentence and vocab)
de_gru_layer = GRU(hsize, return_sequences=True)
de_output_tensors = de_gru_layer(de_input_tensors, initial_state=en_state) # initial_state is connected to the output state of encoder
# Decoder Prediction
de_dense_layer = Dense(fr_vocab, activation= 'softmax' )
de_timed_dense_layer = TimeDistributed( de_dense_layer ) 
de_prediction = de_timed_dense_layer(de_output_tensors)
model_tf = Model(inputs=[en_input_tensors, de_input_tensors], outputs=de_prediction)
model_tf.compile(optimizer='adam', loss="categorical_crossentropy", metrics=["acc"])

## Helper function that takes in text and outputs into padded, embedded, reversed, one-hot vectors 
def sents2seqs(input_type, sentences, onehot=False, pad_type='post', reverse=False):     
    # logic (look into TEXT IN KERAS)
    return preproc_text
    
for ei in range(n_epochs):
    for i in range(0,train_size,bsize):
        encoder_x = sents2seqs('source', tr_en[i:i+bsize], onehot=True, reverse=True) # Input data for encoder
        decoder_xy = sents2seqs('target', tr_fr[i:i+bsize], onehot=True) # Preprocessed data before passing to decoder
        decoder_x = decoder_xy[:,:-1,:] # Inputs - All French words except the last word (onehot encoded)
        decoder_y = decoder_xy[:,1:,:] # Outputs - All French words except the first word as next sequence of input (onehot encoded)
        model_tf.train_on_batch([encoder_x, decoder_x], decoder_y)

### WITH EMBEDDING LAYER

en_input_tensors = Input(shape=(en_len,)) # Input layer will only take in a sentence 
en_embedding_layer = Embedding(en_vocab, 96, input_length=en_len) # Embedding layer will map inputs into en_vocabX96 embedding matrix
en_embedded_tensors = en_embedding_layer(en_input_tensors)
en_gru_layer = GRU(hsize, return_state=True)
en_output_tensors, en_state = en_gru_layer(en_embedded_tensors)

de_input_tensors = Input(shape=(fr_len-1,)) # Input layer will only take in a sentence with 1 less word
de_embedding_layer = Embedding(fr_vocab, 96, input_length=fr_len-1) # this layer will map inputs into fr_vocabX96 embedding matrix
de_embedded_tensors = de_embedding_layer(de_input_tensors)
de_gru_layer = GRU(hsize, return_sequences=True, return_state=True)
de_prediction, de_state = de_gru_layer(de_embedded_tensors, initial_state=en_state) # Connect encoder output state into decoder state

model_emb = Model(inputs=[en_input_tensors, de_input_tensors], outputs=de_prediction)
model_emb.compile(optimizer='adam', loss="categorical_crossentropy", metrics=["acc"])

for ei in range(3): # epoch
    for i in range(0, train_size, bsize): # batch
        encoder_x = sents2seqs('source', tr_en[i:i+bsize], onehot=False, reverse=True) # encoder input data
        decoder_xy = sents2seqs('target', tr_fr[i:i+bsize], onehot=False) # preprocessed data before passing into decoder
        decoder_x = decoder_xy[:,:-1]  # Inputs - All French words except the last word (not onehot encoded, since embedded layer)
        decoder_xy_oh = sents2seqs('target', tr_fr[i:i+bsize], onehot=True) # One-hot encoding the target for getting y for modeling
        decoder_y = decoder_xy_oh[:,1:,:] # Outputs- All French words except the first word as next sequence of input (onehot encoded)
        model_emb.train_on_batch([encoder_x, decoder_x], decoder_y)
        res = model_emb.evaluate([encoder_x, decoder_x], decoder_y, batch_size=bsize, verbose=0)
        print("{} => Loss:{}, Train Acc: {}".format(ei+1,res[0], res[1]*100.0))

# NOTE : Problem with this model, the decoder also needs values that we need to predict. 
# Solution: You need to create an inference model that will generate word by word prediction. You then need to copy this model's each layer's weight to the inference model's subsequent layer's weight
# see DECODER OF THE INFERENCE MODEL IN KERAS

```